In [1]:
import random
import numpy as np
import os
import torch 

def set_seed(seed=24):
    """Setea semilla para reproducibilidad general"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # si usas multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"Semilla fijada en: {seed}")

# Llamar a la función
set_seed(24)

Semilla fijada en: 24


In [2]:
import sys
sys.path.append('../')
 
import pandas as pd 
from sklearn.metrics import cohen_kappa_score, accuracy_score,balanced_accuracy_score
from plotly import express as px
from tutoriales.utils import plot_confusion_matrix, get_artifact_filename
from json import loads
from joblib import load, dump
import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact
from optuna.visualization import plot_param_importances
from optuna.importance import FanovaImportanceEvaluator
from optuna.importance import MeanDecreaseImpurityImportanceEvaluator  
from optuna.importance import PedAnovaImportanceEvaluator
from optuna.visualization import plot_contour

c:\Users\matia\anaconda3\envs\ldi2_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys, numpy.core
# Shim: permite cargar joblib guardados con numpy >= 2.0 en entornos con numpy 1.x
sys.modules.setdefault("numpy._core", numpy.core)
for _sub in ["numeric", "multiarray", "umath", "fromnumeric", "arrayprint", "strings"]:
    mod = getattr(numpy.core, _sub, numpy.core)
    sys.modules.setdefault(f"numpy._core.{_sub}", mod)

In [4]:
# Paths
BASE_DIR = '../'
PATH_TO_MODELS = os.path.join(BASE_DIR, "work/models")
PATH_TO_TRAIN = os.path.join(BASE_DIR, "work/cleaned/train_clean.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

***Carga del modelo Tabular:*** Elección Stacking de modelos.

In [ ]:
#lgb_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_modelo_1__variables_originales.joblib'))
#lgb_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_modelo_2_m0__variables_completas.joblib'))
lgb_dataset = load(os.path.join(PATH_TO_MODELS, 'stacking_modelos_v1.joblib'))




***Carga del modelo de Texto*** 

In [6]:
MODEL_NAME = '01 DistilBert'
MODEL_VERSION = '5.0'

study_bert = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3", 
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)


[I 2026-05-18 13:15:19,276] Using an existing study with name '01 DistilBert_5.0' instead of creating a new one.


In [7]:
bert_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_bert,'test')))

***Carga del modelo de Imágenes***

In [8]:
# Cargar el modelo ResNet
MODEL_NAME_RESNET = '04 ResNet Augment'
MODEL_VERSION_RESNET = '1.0.0'

study_resnet = optuna.create_study(
    direction='maximize',
    storage="sqlite:///../work/optuna_artifacts/db.sqlite3",
    study_name=f'{MODEL_NAME_RESNET}_{MODEL_VERSION_RESNET}',
    load_if_exists=True
)

[I 2026-05-18 13:15:19,434] Using an existing study with name '04 ResNet Augment_1.0.0' instead of creating a new one.


In [9]:
resnet_dataset = load(os.path.join(PATH_TO_TEMP_FILES, 'test_04 ResNet Augment_1.0.0_1.joblib'))
#resnet_dataset = load(os.path.join(PATH_TO_TEMP_FILES,get_artifact_filename(study_resnet,'test')))

In [10]:
merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(bert_dataset[['PetID', 'pred']].rename({'pred':'bert_pred_score'},axis=1),
                  on='PetID', how='outer')

In [11]:
# Unir ResNet al dataframe fusionado
merged_datasets = merged_datasets.merge(
    resnet_dataset[['PetID', 'pred']].rename({'pred': 'resnet_pred_score'}, axis=1),
    on='PetID', how='outer'
)

In [12]:
#merged_datasets.head()

In [ ]:
# Porcentaje de nulos en cada columna
merged_datasets.isnull().mean().mul(100).round(2).rename('% nulos')


PetID                0.00
lgb_pred_score       0.00
AdoptionSpeed        0.00
bert_pred_score      0.10
resnet_pred_score    2.27
Name: % nulos, dtype: float64

In [14]:
# Limpiar nulos (rellenar con arrays de ceros si algún modelo no tiene predicción para un PetID)
merged_datasets['resnet_pred_score'] = [np.zeros(5) if type(i) is float else i for i in merged_datasets['resnet_pred_score']]
merged_datasets['bert_pred_score']   = [np.zeros(5) if type(i) is float else i for i in merged_datasets['bert_pred_score']]
merged_datasets['lgb_pred_score']    = [np.zeros(5) if type(i) is float else i for i in merged_datasets['lgb_pred_score']]


In [ ]:
# Normalizar scores de cada modelo
all_values = [item for sublist in merged_datasets["resnet_pred_score"] for item in sublist]
min_val = min(all_values)
max_val = max(all_values)

def normalizar_lista(lista):
    return [(x - min_val) / (max_val - min_val) for x in lista]

merged_datasets["resnet_pred_score"] = merged_datasets["resnet_pred_score"].apply(normalizar_lista)

all_values1 = [item for sublist in merged_datasets["bert_pred_score"] for item in sublist]
min_val1 = min(all_values1)
max_val1 = max(all_values1)

def normalizar_lista1(lista):
    return [(x - min_val1) / (max_val1 - min_val1) for x in lista]

merged_datasets["bert_pred_score"] = merged_datasets["bert_pred_score"].apply(normalizar_lista1)

all_values2 = [item for sublist in merged_datasets["lgb_pred_score"] for item in sublist]
min_val2 = min(all_values2)
max_val2 = max(all_values2)

def normalizar_lista2(lista):
    return [(x - min_val2) / (max_val2 - min_val2) for x in lista]

merged_datasets["lgb_pred_score"] = merged_datasets["lgb_pred_score"].apply(normalizar_lista2)


In [16]:
#merged_datasets.head()

### Merging de modelos

Para mejorar el poder predictivo de los modelos anteriormente vistos se decidió realizar una combinación linear del aporte de predicción de cada uno optimizando el peso asignado a cada modelo, de esta forma se logra una mejor generalización sobre datos no nuevos o no analizados. La función objetivo a maximizar fue la métrica Kappa. 

In [ ]:
# Optimización con Optuna para 3 pesos
def objective(trial):
    # Definir pesos para los tres modelos
    w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
    w_bert = trial.suggest_float('w_bert', 0.0, 1.0)
    w_resnet = trial.suggest_float('w_resnet', 0.0, 1.0)
    
    # Normalización
    total_w = w_lgb + w_bert + w_resnet
    if total_w == 0: return 0
    
    # Cálculo vectorizado para mayor velocidad
    lgb_scores = np.stack(merged_datasets['lgb_pred_score'].values)
    bert_scores = np.stack(merged_datasets['bert_pred_score'].values)
    resnet_scores = np.stack(merged_datasets['resnet_pred_score'].values)
    
    combined_scores = (
        (w_lgb / total_w) * lgb_scores + 
        (w_bert / total_w) * bert_scores + 
        (w_resnet / total_w) * resnet_scores
    )
    
    preds_final = np.argmax(combined_scores, axis=1)
    
    return cohen_kappa_score(merged_datasets['AdoptionSpeed'], preds_final, weights='quadratic')

In [18]:
# Ejecutar el estudio
STORAGE_URL = "sqlite:///../work/db-blend.sqlite3"
study_blend = optuna.create_study(
    direction='maximize',
    storage=STORAGE_URL,
    study_name="Ensemble_stacking_BERT_ResNet",
    load_if_exists=True
)
study_blend.optimize(objective, n_trials=100)

[I 2026-05-18 13:15:19,852] A new study created in RDB with name: Ensemble_stacking_BERT_ResNet
[I 2026-05-18 13:15:19,989] Trial 0 finished with value: 0.3673987074877929 and parameters: {'w_lgb': 0.5623810873635466, 'w_bert': 0.5597018168477498, 'w_resnet': 0.7282818335393161}. Best is trial 0 with value: 0.3673987074877929.
[I 2026-05-18 13:15:20,083] Trial 1 finished with value: 0.2962559839611332 and parameters: {'w_lgb': 0.37674347326049507, 'w_bert': 0.8300189579021787, 'w_resnet': 0.11652682220924504}. Best is trial 0 with value: 0.3673987074877929.
[I 2026-05-18 13:15:20,175] Trial 2 finished with value: 0.40119174798330437 and parameters: {'w_lgb': 0.9902558490028281, 'w_bert': 0.4461965027752285, 'w_resnet': 0.671051054781868}. Best is trial 2 with value: 0.40119174798330437.
[I 2026-05-18 13:15:20,264] Trial 3 finished with value: 0.40294151635618336 and parameters: {'w_lgb': 0.7828851148088557, 'w_bert': 0.40919732077450655, 'w_resnet': 0.8350236529914019}. Best is trial 3

In [19]:
# Resultados
best_params = study_blend.best_params
sum_best_w = sum(best_params.values())

print(f"Mejor Kappa: {study_blend.best_value:.4f}")
print(f"Pesos óptimos: {best_params}")

Mejor Kappa: 0.4343
Pesos óptimos: {'w_lgb': 0.5708432690962936, 'w_bert': 0.13805008046605272, 'w_resnet': 0.9490744755931527}


In [20]:
best_kappa = study_blend.best_trial.value
print(f"Mejor puntuación Kappa: {best_kappa}")

Mejor puntuación Kappa: 0.43434500540835486


In [21]:
# Crear la columna de predicción final optimizada

def _prediction_vector(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=float)
    return np.zeros(5, dtype=float)

merged_datasets['blend_pred_score'] = [
    (best_params['w_lgb'] / sum_best_w) * _prediction_vector(r['lgb_pred_score']) +
    (best_params['w_bert'] / sum_best_w) * _prediction_vector(r['bert_pred_score']) +
    (best_params['w_resnet'] / sum_best_w) * _prediction_vector(r['resnet_pred_score'])
    for _, r in merged_datasets.iterrows()
]

In [22]:
#merged_datasets[['lgb_pred_score', 'bert_pred_score', 'resnet_pred_score']]
merged_datasets['blend_pred_score']

0       [0.23478905172868395, 0.49208459419191425, 0.5...
1       [0.20852314915415943, 0.43071336616743316, 0.3...
2       [0.19402283242057594, 0.41175530962245604, 0.5...
3       [0.0846611825981198, 0.2647441212410391, 0.437...
4       [0.11841994483416847, 0.39395605370627373, 0.5...
                              ...                        
2994    [0.16442829925739819, 0.37737929388960817, 0.3...
2995    [0.15470517453291774, 0.419187455477587, 0.508...
2996    [0.13530148668056566, 0.5796132348901314, 0.52...
2997    [0.2048073742890255, 0.44625913972248854, 0.43...
2998    [0.1307515874859945, 0.46386738192800814, 0.50...
Name: blend_pred_score, Length: 2999, dtype: object

In [23]:
#merged_datasets['blend_pred_score']

In [24]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['lgb_pred_score'].apply(np.argmax), 
                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['lgb_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))

In [25]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['bert_pred_score'].apply(np.argmax), 
                    title = 'Bert Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['bert_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


In [26]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['resnet_pred_score'].apply(np.argmax), 
                    title = 'ResNet Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['resnet_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


In [27]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['blend_pred_score'].apply(np.argmax), 
                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                   merged_datasets['blend_pred_score'].apply(np.argmax), 
                                                                    weights='quadratic')))


### Conclusión

Como puede observarse en cada matriz de confusión donde se indica el Kappa obtenido por cada modelo, para el modelo blended el valor de Kappa es superior a los anteriores, llegando al valor de 0.4343.
